# Early American HTR — Public Competition Training & Eval (Colab Pro)

**Competition-cleared public data only (`CC-BY-4.0` and `Public Domain`).**

This notebook mirrors our high-resolution, trainable-vision, sham-controlled training harness while strictly restricting datasets and models to competition-eligible open resources:

1. **Data (100% Public & Rights-Cleared):**
   - **Revolutionary City HTR Corpus** (Zenodo `15776323`, **CC-BY-4.0**) downloaded directly to Colab SSD (pairing `.jpg`/`.jpeg` and `.png` pages with ALTO-XML), parsed in annotated ALTO reading order via `eahtr.corpus.parse_alto_xml_lines(..., sort_regions=False)`, cleaned via `eahtr.hygiene.clean_diplomatic_target`, grouped via `assign_revcity_item_groups(max_id_gap=30)` + `REVCITY_COLLECTION_OVERRIDES`, and split **before** smoke subsampling with hard-gated zero document-group, writer, image SHA-256, `>= 8`-word exact-text, or 32-word n-gram leakage (plus advisory perceptual dHash review).
   - **Optional Library of Congress *By the People*** (`LCCN 2020446971`, **Public Domain**) behind `INCLUDE_LOC_BY_THE_PEOPLE = False` as a genre-shift challenge panel (`hard_case_panel.jsonl` also captures blanks/leader-dots/duplicates).
2. **Candidate Apache-2.0 Models:**
   - `gemma4_12b` (`google/gemma-4-12B-it`, **primary submission candidate**, ungated) — 12B dense unified VLM (`gemma4_unified`, `1120` soft tokens; `Stage 1 SFT` → conditional `Stage 2 VLM SMPO` with `chosen_sft_ratio: 1.0`).
   - Halo cluster track also tests conditional `thinkingmachines/Inkling-2-Small` (`EP=8`, frozen-vision image-conditioned pilot) after Dec 1.
3. **Dev Discipline, Evaluation & Stage 2 Rollouts:**
   - Development runs score **validation (`val`) ONLY** (with hash-keyed caches, same-checkpoint image-swap checks, strict/lenient + newline/line-boundary + order-tolerant CER, dual-convention zero-shot scoring, and verifiable `eahtr.rewards`). `test_public` remains frozen until final checkpoint freeze.
   - Optional Cell 7 generates and scores $K=4$ multi-temperature rollouts from the merged SFT model on strictly held-out `oof_train` folds for Halo Stage 2 **VLM SMPO** (`chosen_sft_ratio: 1.0`).

> **Halo Competition Rule Note (Terms §§ 4.2 & 5):** Pre-Dec-1 Colab runs establish baseline benchmarks, data hygiene, vision-gradient flow, and VLM preference rollouts on the competition corpus. Final competition submissions after Dec 1 must be reproduced by the official **Halo** training recipe (`github.com/whitecircle/halo`) under published Track Rules.

In [ ]:
!pip install -q "transformers==5.23.0" "peft==0.18.1" "accelerate>=1.11.0,<1.12.0" "bitsandbytes>=0.49.2,<0.50.0" "rapidfuzz>=3.9.0" "pyyaml>=6.0" "tqdm>=4.66"

import csv
import gc
import hashlib
import io
import json
import math
import os
import random
import re
import subprocess
import tarfile
import time
import zipfile
from collections import Counter
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

import torch
import transformers
from PIL import Image
from rapidfuzz.distance import Levenshtein
from tqdm.auto import tqdm
from huggingface_hub import login as hf_login
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
AutoModelForMultimodalLM = getattr(transformers, 'AutoModelForMultimodalLM', None)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as exc:
        raise RuntimeError(f'Google Drive mount failed while USE_GOOGLE_DRIVE=True: {exc}') from exc
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Google Drive mount did not expose /content/drive/MyDrive while USE_GOOGLE_DRIVE=True')

REPO_URL = 'https://github.com/markandster/early-american-htr-halo.git'
REPO_DIR = Path('/content/early-american-htr-halo')
PINNED_REPO_COMMIT = ''
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
if PINNED_REPO_COMMIT:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', PINNED_REPO_COMMIT], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', PINNED_REPO_COMMIT], check=True)
subprocess.run(['pip', 'install', '-q', '--no-deps', '-e', str(REPO_DIR)], check=True)
import sys
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))
REPO_GIT_SHA = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'Repo commit: {REPO_GIT_SHA} | transformers={transformers.__version__}')

from eahtr.colab_pipeline import (
    COMPETITION_CANDIDATES,
    DIPLOMATIC_PROMPT,
    LOC_WASHINGTON_ITEM_JSON_URL,
    REVCITY_ZENODO_BYTES,
    REVCITY_ZENODO_MD5,
    REVCITY_ZENODO_URL,
    assign_balanced_group_splits,
    assign_deterministic_split,
    assign_revcity_item_groups,
    make_sham_derangement,
    parse_revcity_stem_id,
    resolve_vision_modules_to_save,
    should_use_4bit,
    validate_competition_candidate,
)
from eahtr.corpus import detect_two_page_opening, filter_corpus_record, parse_alto_xml_lines, parse_page_xml_lines
from eahtr.halo_export import verify_exported_vlm_dir
from eahtr.hf_dataset import push_splits_to_hf_hub
from eahtr.hygiene import audit_split_disjointness, clean_diplomatic_target, compute_dhash, extract_word_ngrams
from eahtr.metrics import normalize_preserve_newlines, normalize_strict, paired_bootstrap_ci, score_line_breaks, score_pair
from eahtr.rewards import build_preference_pairs, recommended_max_new_tokens, score_group_completions, score_transcription

assert torch.cuda.is_available(), 'CUDA GPU required (select L4 24GB, A100 40GB/80GB, or H100 80GB in Colab Pro)'
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
print(f'GPU: {GPU_NAME} | VRAM: {VRAM_GB} GB | PyTorch: {torch.__version__}')

hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        hf_token = None
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    hf_login(token=hf_token, add_to_git_credential=False)
    print('Hugging Face authenticated via HF_TOKEN.')
else:
    print(
        'NOTE: HF_TOKEN not set. All active Apache-2.0 base checkpoints '
        '(google/gemma-4-12B-it, thinkingmachines/Inkling-2-Small) are ungated; '
        'HF_TOKEN is only required when PUBLISH_TO_HF_DATASET=True.'
    )

In [ ]:
# === Run Configuration (100% Public Competition Data Only) ===
OUTPUT_ROOT = (
    Path('/content/drive/MyDrive/early-american-htr-halo-runs')
    if USE_GOOGLE_DRIVE
    else Path('/content/eahtr_runs')
)
WORK_ROOT = Path('/content/eahtr_work')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Set SMOKE_TEST_MODE = True for a ~15-25 min end-to-end validation run first;
# switch to False for the full 3,145-page active public dataset run (2,897 train / 124 val / 124 test_public).
SMOKE_TEST_MODE = True

# Select which Apache-2.0 competition candidates to run:
# Available keys: ['gemma4_12b']
CANDIDATE_KEYS = ['gemma4_12b']
ARMS = ['baseline_zeroshot', 'real', 'sham_short']
# Smoke mode matches `real` (128 pages); full-mode Colab `sham_short` is a short unmatched control (<= 512 pages); matched-compute sham runs in Halo.
SHAM_MAX_PRESENTATIONS = 128 if SMOKE_TEST_MODE else 512
CHECKPOINT_EVERY_OPT_STEPS = 50

# Public rights-cleared datasets (CC-BY-4.0 / Public Domain ONLY)
INCLUDE_REVCITY = True
REVCITY_MAX_PAGES = 400 if SMOKE_TEST_MODE else None
VAL_FRACTION = 0.05
TEST_FRACTION = 0.05
MAX_EVAL_GROUP_PAGES = 40
OOF_NUM_FOLDS = 1
OOF_FOLD_IDX = 0
PORTAL_CREATOR_MAP_PATH = ''  # Optional JSON mapping revcity stem/item IDs to creator/hand IDs

# Optional Public Domain LOC By the People (LCCN 2020446971) challenge eval
INCLUDE_LOC_BY_THE_PEOPLE = False
LOC_MAX_PAGES = 150  # Polite cap when enabled

# Training hyperparameters (full-resolution trainable-vision SFT setup)
SEED = 20261125
EPOCHS = 1 if SMOKE_TEST_MODE else 2
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 1e-4
VISION_LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
GRAD_ACCUM_STEPS = 8
MAX_SEQ_LEN = 4096
MAX_EVAL_NEW_TOKENS = 2048
VAL_EVAL_MAX_PAGES = 32 if SMOKE_TEST_MODE else 128
FINAL_FREEZE_EVAL_TEST_PUBLIC = False  # Keep False during dev; test_public is frozen for final eval only

# Optional Hugging Face Hub Dataset export (strictly public/CC-BY-4.0 splits only)
PUBLISH_TO_HF_DATASET = False
HF_DATASET_REPO_ID = 'markandster/early-american-htr-public'

# Stage 2 VLM SMPO / Offline GRPO rollout generation (Cell 7)
GENERATE_OFFLINE_GRPO_ROLLOUTS = False
ROLLOUT_SOURCE = 'oof_train'
ALLOW_IN_SAMPLE_TRAIN_ROLLOUTS = False
GRPO_ROLLOUT_PROMPTS = 64
GRPO_NUM_GENERATIONS = 4

for k in CANDIDATE_KEYS:
    validate_competition_candidate(COMPETITION_CANDIDATES[k])
print('SMOKE_TEST_MODE:', SMOKE_TEST_MODE, '| candidates:', CANDIDATE_KEYS, '| arms:', ARMS)

In [ ]:
# === Step 1: Download, Clean & Split Rights-Cleared Public Datasets ===
def _file_md5(path: Path) -> str:
    h = hashlib.md5()
    with path.open('rb') as fh:
        for chunk in iter(lambda: fh.read(4 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def download_with_retry(
    url: str,
    dest: Path,
    max_attempts: int = 4,
    min_existing_bytes: int = 1_000_000_000,
    expected_bytes: int | None = None,
    expected_md5: str | None = None,
) -> None:
    if dest.exists() and dest.stat().st_size >= min_existing_bytes:
        size_ok = expected_bytes is None or dest.stat().st_size == expected_bytes
        md5_ok = expected_md5 is None or _file_md5(dest) == expected_md5
        if size_ok and md5_ok:
            print(f'Reusing verified archive: {dest} ({dest.stat().st_size / 1e6:.1f} MB)')
            return
        print(f'Cached archive {dest} failed size/MD5 check; re-downloading...')
        dest.unlink()
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + '.part')
    last_err = None
    for attempt in range(1, max_attempts + 1):
        try:
            existing = part.stat().st_size if part.exists() else 0
            headers = {'User-Agent': 'early-american-htr-halo/0.1 (public-corpus-research)'}
            if existing > 0:
                headers['Range'] = f'bytes={existing}-'
            req = Request(url, headers=headers)
            with urlopen(req, timeout=120) as resp:
                status = getattr(resp, 'status', 200)
                mode = 'ab' if (existing > 0 and status == 206) else 'wb'
                total = int(resp.headers.get('Content-Length') or 0) + (existing if mode == 'ab' else 0)
                with part.open(mode) as fh, tqdm(
                    total=total or None,
                    initial=existing if mode == 'ab' else 0,
                    unit='B',
                    unit_scale=True,
                    desc=dest.name,
                ) as pbar:
                    while True:
                        chunk = resp.read(4 << 20)
                        if not chunk:
                            break
                        fh.write(chunk)
                        pbar.update(len(chunk))
            part.replace(dest)
            if expected_bytes is not None and dest.stat().st_size != expected_bytes:
                actual = dest.stat().st_size
                dest.unlink(missing_ok=True)
                raise OSError(f'Download byte count mismatch for {dest}: {actual} != {expected_bytes}')
            if expected_md5 is not None and _file_md5(dest) != expected_md5:
                dest.unlink(missing_ok=True)
                raise OSError(f'Download MD5 mismatch for {dest}')
            return
        except (HTTPError, URLError, OSError, TimeoutError) as exc:
            last_err = exc
            wait_s = min(60, 5 * attempt)
            print(f'[Attempt {attempt}/{max_attempts}] Download error ({type(exc).__name__}: {exc}); retrying in {wait_s}s...')
            time.sleep(wait_s)
    raise RuntimeError(f'Failed to download {url}: {last_err}')


def fetch_bytes_polite(url: str, max_attempts: int = 3, delay_s: float = 0.5) -> bytes:
    last_err = None
    for attempt in range(1, max_attempts + 1):
        try:
            time.sleep(delay_s)
            req = Request(url, headers={'User-Agent': 'early-american-htr-halo/0.1 (public-corpus-research)'})
            with urlopen(req, timeout=60) as resp:
                return resp.read()
        except (HTTPError, URLError, OSError, TimeoutError) as exc:
            last_err = exc
            time.sleep(min(30, 3 * attempt))
    raise RuntimeError(f'Failed GET {url}: {last_err}')


def _stratified_smoke_slice(rows_by_split: dict, max_total: int, seed: int) -> dict:
    total = sum(len(v) for v in rows_by_split.values())
    if total <= max_total:
        return rows_by_split
    rng = random.Random(seed)
    out = {'train': [], 'val': [], 'test_public': []}
    min_eval = min(24, max(8, max_total // 10))
    budgets = {
        'val': min(len(rows_by_split['val']), max(min_eval, int(max_total * VAL_FRACTION))),
        'test_public': min(len(rows_by_split['test_public']), max(min_eval, int(max_total * TEST_FRACTION))),
    }
    budgets['train'] = min(len(rows_by_split['train']), max(1, max_total - budgets['val'] - budgets['test_public']))
    for split, k in budgets.items():
        by_group = {}
        for item in rows_by_split[split]:
            gid = item[6].get('document_id') or item[1]
            by_group.setdefault(gid, []).append(item)
        gkeys = list(by_group)
        rng.shuffle(gkeys)
        for gk in gkeys:
            rng.shuffle(by_group[gk])
        picked = []
        while len(picked) < k and any(by_group.values()):
            for gk in gkeys:
                if by_group[gk]:
                    picked.append(by_group[gk].pop())
                    if len(picked) >= k:
                        break
        out[split] = picked
    return out


all_records_by_split = {'train': [], 'val': [], 'test_public': []}
hard_case_rows = []
loc_challenge_rows = []
corpus_report = Counter()
creator_map = {}
if PORTAL_CREATOR_MAP_PATH and Path(PORTAL_CREATOR_MAP_PATH).is_file():
    creator_map = json.loads(Path(PORTAL_CREATOR_MAP_PATH).read_text(encoding='utf-8'))

if INCLUDE_REVCITY:
    rc_dir = WORK_ROOT / 'revcity'
    rc_zip = rc_dir / 'revcity-htr.zip'
    rc_pages = rc_dir / 'pages'
    rc_pages.mkdir(parents=True, exist_ok=True)
    download_with_retry(
        REVCITY_ZENODO_URL,
        rc_zip,
        min_existing_bytes=1_000_000_000,
        expected_bytes=REVCITY_ZENODO_BYTES,
        expected_md5=REVCITY_ZENODO_MD5,
    )

    with zipfile.ZipFile(rc_zip) as zf:
        by_stem = {}
        for member in zf.namelist():
            low = member.lower()
            stem = Path(member).stem
            if low.endswith(('.jpg', '.jpeg', '.png')):
                by_stem.setdefault(stem, {})['img'] = member
            elif low.endswith('.xml'):
                by_stem.setdefault(stem, {})['xml'] = member
        paired_stems = sorted(s for s, d in by_stem.items() if 'img' in d and 'xml' in d)
        print(f'Scanning all {len(paired_stems)} paired (JPG/PNG + ALTO-XML) pages in Zenodo 15776323 (CC-BY-4.0)')

        # Assign volume/item groups (max_id_gap=30) & merge exact/n-gram linked keys before splitting
        item_groups = assign_revcity_item_groups(paired_stems, max_id_gap=30, creator_map=creator_map or None)
        admitted_items = []
        for stem in paired_stems:
            entry = by_stem[stem]
            xml_str = zf.read(entry['xml']).decode('utf-8', errors='replace')
            try:
                lines = parse_alto_xml_lines(xml_str, sort_regions=False)
            except Exception:
                corpus_report['revcity_xml_parse_error'] += 1
                continue
            raw_text = '\n'.join(lines)
            accepted, reason = filter_corpus_record({'source_name': 'Revolutionary City', 'license': 'CC-BY-4.0', 'text': raw_text})
            cleaned_text = clean_diplomatic_target(raw_text)
            info = item_groups.get(stem, {})
            safe_id = info.get('safe_id') or stem.replace('%3A', '_')
            if not accepted or len(cleaned_text.split()) < 8:
                drop_tag = reason if not accepted else 'dropped_under_8_words'
                corpus_report[f'revcity_{drop_tag}'] += 1
                img_ext = Path(entry['img']).suffix.lower()
                hc_img = rc_pages / f'hard_{safe_id}{img_ext}'
                if not hc_img.exists():
                    hc_img.write_bytes(zf.read(entry['img']))
                has_empty_alto = (len(raw_text.strip()) == 0)
                gt_status = 'missing_or_unverified_gt' if has_empty_alto else ('empty_after_hygiene' if not cleaned_text.strip() else 'nonblank')
                hard_case_rows.append({
                    'record_id': f'revcity-{safe_id}',
                    'document_id': info.get('document_id') or f'revcity:{safe_id}',
                    'writer_key': info.get('writer_key') or f'unattributed_item:{safe_id}',
                    'image_path': str(hc_img),
                    'reason': drop_tag,
                    'has_empty_alto_text': has_empty_alto,
                    'ground_truth_status': gt_status,
                    'raw_text': raw_text,
                    'text': cleaned_text or raw_text,
                })
                continue
            base_key = info['writer_key'] if info.get('hand_attributed') == 'true' else (info.get('document_id') or f'revcity:{safe_id}')
            admitted_items.append((stem, safe_id, entry['img'], xml_str, raw_text, cleaned_text, info, base_key))

        # Union-find transitive closure across 32-gram & >=8-word exact-text matches so related docs never straddle splits
        uf_parent = {it[7]: it[7] for it in admitted_items}
        def _uf_find(k):
            while uf_parent[k] != k:
                uf_parent[k] = uf_parent[uf_parent[k]]
                k = uf_parent[k]
            return k
        def _uf_union(a, b):
            ra, rb = _uf_find(a), _uf_find(b)
            if ra != rb:
                uf_parent[max(ra, rb)] = min(ra, rb)
        seen_sig = {}
        for _, _, _, _, _, c_txt, _, b_key in admitted_items:
            sigs = set(extract_word_ngrams(c_txt, n=32))
            words_norm = ' '.join(c_txt.lower().split())
            if len(words_norm.split()) >= 8:
                sigs.add('exact:' + hashlib.sha256(words_norm.encode('utf-8')).hexdigest())
            for sig in sigs:
                if sig in seen_sig:
                    _uf_union(seen_sig[sig], b_key)
                else:
                    seen_sig[sig] = b_key

        group_stats = {}
        for _, _, _, _, _, _, info, b_key in admitted_items:
            root_g = _uf_find(b_key)
            ent = group_stats.setdefault(root_g, {'pages': 0, 'institution': info.get('institution') or 'revcity'})
            ent['pages'] += 1
        split_map = assign_balanced_group_splits(
            group_stats,
            val_fraction=VAL_FRACTION,
            test_fraction=TEST_FRACTION,
            max_eval_group_pages=MAX_EVAL_GROUP_PAGES,
            seed=SEED,
        )
        split_buckets = {'train': [], 'val': [], 'test_public': []}
        for stem, safe_id, img_m, xml_str, raw_text, cleaned_text, info, base_key in admitted_items:
            split = split_map[_uf_find(base_key)]
            split_buckets[split].append((stem, safe_id, img_m, xml_str, raw_text, cleaned_text, info))

        if REVCITY_MAX_PAGES is not None:
            split_buckets = _stratified_smoke_slice(split_buckets, REVCITY_MAX_PAGES, SEED)

        for split, items in split_buckets.items():
            for stem, safe_id, img_member, xml_str, raw_text, cleaned_text, info in tqdm(
                items, desc=f'Extract RevCity {split}'
            ):
                img_ext = Path(img_member).suffix.lower()
                img_bytes = zf.read(img_member)
                img_sha = hashlib.sha256(img_bytes).hexdigest()
                img_out = rc_pages / f'{safe_id}{img_ext}'
                if not img_out.exists():
                    img_out.write_bytes(img_bytes)
                dhash_hex = compute_dhash(img_out)
                all_records_by_split[split].append({
                    'record_id': f'revcity-{safe_id}',
                    'collection': 'revcity-zenodo-15776323',
                    'document_id': info.get('document_id') or f'revcity:{safe_id}',
                    'writer_key': info.get('writer_key') or f'unattributed_item:{safe_id}',
                    'writer_attributed': info.get('hand_attributed') == 'true',
                    'is_two_page_opening': detect_two_page_opening(xml_str),
                    'image_format': img_ext.lstrip('.'),
                    'image_path': str(img_out),
                    'image_sha256': img_sha,
                    'image_dhash': dhash_hex,
                    'raw_text': raw_text,
                    'text': cleaned_text,
                    'word_count': len(cleaned_text.split()),
                    'license': 'CC-BY-4.0',
                    'split': split,
                })
                corpus_report[f'revcity_admitted_{split}'] += 1
                corpus_report[f'revcity_ext_{img_ext.lstrip(".")}'] += 1

if INCLUDE_LOC_BY_THE_PEOPLE:
    loc_dir = WORK_ROOT / 'loc_btp'
    loc_pages = loc_dir / 'pages'
    loc_pages.mkdir(parents=True, exist_ok=True)
    item_meta = json.loads(fetch_bytes_polite(LOC_WASHINGTON_ITEM_JSON_URL, delay_s=0.2).decode('utf-8'))
    zip_urls = [
        sub['url']
        for res in item_meta.get('resources', [])
        for f_entry in res.get('files', [])
        for sub in (f_entry if isinstance(f_entry, list) else [f_entry])
        if isinstance(sub, dict) and str(sub.get('url', '')).endswith('.zip')
    ]
    assert zip_urls, 'Could not find Public Domain dataset .zip URL in LOC item metadata'
    loc_zip = loc_dir / '2020446971.zip'
    download_with_retry(zip_urls[0], loc_zip, min_existing_bytes=50_000)

    loc_added = 0
    with zipfile.ZipFile(loc_zip) as zf:
        csv_names = [n for n in zf.namelist() if n.lower().endswith('.csv')]
        assert csv_names, 'No CSV found inside LOC By the People zip'
        reader = csv.DictReader(io.StringIO(zf.read(csv_names[0]).decode('utf-8', errors='replace')))
        for row_idx, row in enumerate(reader):
            if loc_added >= LOC_MAX_PAGES:
                break
            item_title = (row.get('Item') or row.get('Title') or '').lower()
            if 'farm report' in item_title:
                corpus_report['loc_skipped_farm_report'] += 1
                continue
            raw_text = row.get('Transcription') or row.get('Text') or ''
            img_url = row.get('DownloadUrl') or row.get('Asset') or ''
            if not raw_text.strip() or not img_url.startswith('http'):
                corpus_report['loc_skipped_missing_fields'] += 1
                continue
            accepted, reason = filter_corpus_record({'source_name': 'LOC By the People', 'license': 'Public Domain', 'text': raw_text})
            cleaned_text = clean_diplomatic_target(raw_text)
            if not accepted or len(cleaned_text.split()) < 8:
                hard_case_rows.append({'record_id': f'loc-btp-{row_idx:05d}', 'reason': reason if not accepted else 'under_8_words', 'raw_text': raw_text})
                continue
            try:
                jpg_bytes = fetch_bytes_polite(img_url, delay_s=0.4)
            except Exception:
                corpus_report['loc_image_fetch_error'] += 1
                continue
            rec_id = f'loc-btp-{row_idx:05d}'
            img_out = loc_pages / f'{rec_id}.jpg'
            img_out.write_bytes(jpg_bytes)
            doc_group = str(row.get('ItemId') or row.get('Item') or rec_id)
            loc_challenge_rows.append({
                'record_id': rec_id,
                'collection': 'loc-by-the-people-2020446971',
                'document_id': f'loc:{doc_group}',
                'writer_key': f'loc_item:{doc_group}',
                'writer_attributed': False,
                'image_path': str(img_out),
                'image_sha256': hashlib.sha256(jpg_bytes).hexdigest(),
                'image_dhash': compute_dhash(img_out),
                'raw_text': raw_text,
                'text': cleaned_text,
                'word_count': len(cleaned_text.split()),
                'license': 'Public Domain',
                'split': 'challenge_loc',
            })
            corpus_report['loc_challenge_admitted'] += 1
            loc_added += 1

train_rows = all_records_by_split['train']
val_rows = all_records_by_split['val']
test_public_rows = all_records_by_split['test_public']
assert len(train_rows) >= 1 and len(val_rows) >= 8 and len(test_public_rows) >= 8, (
    f'Splits too small (check grouping/flags): train={len(train_rows)}, val={len(val_rows)}, test={len(test_public_rows)}'
)

val_audit = audit_split_disjointness(train_rows, val_rows, ngram_size=32, max_dhash_distance=4)
test_audit = audit_split_disjointness(train_rows, test_public_rows, ngram_size=32, max_dhash_distance=4)
val_test_audit = audit_split_disjointness(val_rows, test_public_rows, ngram_size=32, max_dhash_distance=4)
assert val_audit['passed'], f'Train/Val leakage detected: {val_audit}'
assert test_audit['passed'], f'Train/Test leakage detected: {test_audit}'
assert val_test_audit['passed'], f'Val/Test leakage detected: {val_test_audit}'

train_doc_ids = {r['document_id'] for r in train_rows}
val_doc_ids = {r['document_id'] for r in val_rows}
test_doc_ids = {r['document_id'] for r in test_public_rows}
hard_case_held_out = []
for hc in hard_case_rows:
    did = hc.get('document_id')
    if did in test_doc_ids:
        hc['split_overlap'] = 'test_public'
        hc['panel_role'] = 'diagnostic_overlaps_test'
    elif did in train_doc_ids:
        hc['split_overlap'] = 'train'
        hc['panel_role'] = 'diagnostic_overlaps_train'
    elif did in val_doc_ids:
        hc['split_overlap'] = 'val'
        hc['panel_role'] = 'held_out_from_train_val_doc'
        hard_case_held_out.append(hc)
    else:
        hc['split_overlap'] = 'none'
        hc['panel_role'] = 'held_out_from_train_and_test'
        hard_case_held_out.append(hc)

manifest_dir = OUTPUT_ROOT / 'manifests'
manifest_dir.mkdir(parents=True, exist_ok=True)
all_manifests = {
    **all_records_by_split,
    'hard_case_panel': hard_case_rows,
    'hard_case_held_out': hard_case_held_out,
    'challenge_loc': loc_challenge_rows,
}
for split_name, rows in all_manifests.items():
    (manifest_dir / f'{split_name}.jsonl').write_text(
        '\n'.join(json.dumps(r, ensure_ascii=False) for r in rows) + ('\n' if rows else ''),
        encoding='utf-8',
    )

def _split_stats(rows):
    return {
        'pages': len(rows),
        'documents': len({r['document_id'] for r in rows}),
        'attributed_hands': len({r['writer_key'] for r in rows if r.get('writer_attributed')}),
        'unattributed_pages': sum(1 for r in rows if not r.get('writer_attributed')),
        'two_page_openings': sum(1 for r in rows if r.get('is_two_page_opening')),
        'words': sum(r['word_count'] for r in rows),
        'collections': dict(Counter(r['collection'] for r in rows)),
    }

audit_summary = {
    'counts': dict(corpus_report),
    'splits': {k: _split_stats(v) for k, v in all_records_by_split.items()},
    'hard_case_panel_pages': len(hard_case_rows),
    'challenge_loc_pages': len(loc_challenge_rows),
    'dhash_review_candidates': (
        val_audit['perceptual_image_collisions']
        + test_audit['perceptual_image_collisions']
        + val_test_audit['perceptual_image_collisions']
    ),
    'val_disjoint_audit': val_audit,
    'test_disjoint_audit': test_audit,
    'val_test_disjoint_audit': val_test_audit,
}
(OUTPUT_ROOT / 'corpus_audit_report.json').write_text(json.dumps(audit_summary, indent=2), encoding='utf-8')
print(json.dumps(audit_summary['splits'], indent=2))
print('Disjointness checks (doc, group, sha256, exact, 32-gram) PASSED; dHash review candidates:', len(audit_summary['dhash_review_candidates']))

if PUBLISH_TO_HF_DATASET:
    print(f'Publishing validated public splits to Hugging Face Hub: {HF_DATASET_REPO_ID} ...')
    !pip install -q datasets huggingface_hub
    ds_url = push_splits_to_hf_hub(all_records_by_split, HF_DATASET_REPO_ID, audit_summary, private=False)
    print(f'Published public dataset: {ds_url}')

In [ ]:
# === Training, Vision-Grad Verification & Verifiable Scoring Loop ===
def load_processor_fullres(spec_dict: dict):
    hf_repo = spec_dict['hf_repo']
    max_pixels = spec_dict['max_pixels']
    trust = bool(spec_dict.get('trust_remote_code', False))
    rev = spec_dict.get('hf_revision')
    proc_kw = {'trust_remote_code': trust}
    if rev:
        proc_kw['revision'] = rev
    try:
        proc = AutoProcessor.from_pretrained(hf_repo, max_pixels=max_pixels, **proc_kw)
    except TypeError:
        proc = AutoProcessor.from_pretrained(hf_repo, **proc_kw)
    ip = getattr(proc, 'image_processor', None)
    if ip is not None:
        if hasattr(ip, 'max_soft_tokens'):
            ip.max_soft_tokens = 1120
        if hasattr(ip, 'image_seq_length'):
            ip.image_seq_length = 1120
        if hasattr(ip, 'max_pixels'):
            ip.max_pixels = max_pixels
    if hasattr(proc, 'image_seq_length'):
        proc.image_seq_length = 1120
    if 'gemma-4' in hf_repo.lower():
        tpl_path = REPO_DIR / 'configs' / 'halo' / 'templates' / 'gemma4_htr_nothink.jinja'
        if tpl_path.is_file():
            htr_tpl = tpl_path.read_text(encoding='utf-8')
            proc.chat_template = htr_tpl
            if hasattr(proc, 'tokenizer') and proc.tokenizer is not None:
                proc.tokenizer.chat_template = htr_tpl
    return proc


def load_vlm_model(hf_repo: str, load_kwargs: dict):
    # Load VLM cleanly; Gemma 4 global_head_dim=512 requires sdpa (FlashAttention rejects it).
    resolved_kwargs = dict(load_kwargs)
    if 'gemma-4' in hf_repo.lower() and 'attn_implementation' not in resolved_kwargs:
        resolved_kwargs['attn_implementation'] = 'sdpa'
    prefer_multimodal = 'inkling' in hf_repo.lower() and AutoModelForMultimodalLM is not None
    loaders = (
        [AutoModelForMultimodalLM, AutoModelForImageTextToText]
        if prefer_multimodal
        else [cls for cls in (AutoModelForImageTextToText, AutoModelForMultimodalLM) if cls is not None]
    )
    last_err = None
    for cls in loaders:
        try:
            return cls.from_pretrained(hf_repo, **resolved_kwargs)
        except (ValueError, KeyError, TypeError) as exc:
            last_err = exc
    raise last_err


def _apply_chat_nothink(processor, messages: list[dict], *, add_generation_prompt: bool) -> str:
    try:
        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=False,
        )
    except TypeError:
        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )


def _resolve_turn_terminator(tokenizer) -> str:
    eot = getattr(tokenizer, 'eot_token', None)
    if isinstance(eot, str) and eot:
        return eot
    vocab = tokenizer.get_vocab() if hasattr(tokenizer, 'get_vocab') else {}
    if '<turn|>' in vocab:
        return '<turn|>'
    return getattr(tokenizer, 'eos_token', None) or ''


def build_training_batch(processor, image: Image.Image, target_text: str, device):
    tok = getattr(processor, 'tokenizer', processor)
    user_msgs = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': DIPLOMATIC_PROMPT}]}]
    prompt_str = _apply_chat_nothink(processor, user_msgs, add_generation_prompt=True)
    eot_token = _resolve_turn_terminator(tok)
    full_str = prompt_str + target_text + eot_token

    prompt_inputs = processor(text=[prompt_str], images=[image], return_tensors='pt')
    full_inputs = processor(text=[full_str], images=[image], return_tensors='pt')
    prompt_len = prompt_inputs['input_ids'].shape[1]

    # Never truncate expanded VLM tensors mid-sample (desyncs mm_token_type_ids / token_type_ids).
    if full_inputs['input_ids'].shape[1] > MAX_SEQ_LEN:
        return None

    assert full_inputs['input_ids'].shape[1] >= prompt_len and torch.equal(
        full_inputs['input_ids'][:, :prompt_len], prompt_inputs['input_ids']
    ), 'Tokenized prompt/full prefix mismatch detected; refusing misaligned training batch.'

    labels = full_inputs['input_ids'].clone()
    labels[:, :prompt_len] = -100
    pad_id = getattr(tok, 'pad_token_id', None)
    if pad_id is not None:
        labels[labels == pad_id] = -100
    img_tid = getattr(tok, 'image_token_id', None)
    if img_tid is not None:
        labels[labels == img_tid] = -100
    assert int((labels[:, prompt_len:] != -100).sum().item()) > 0, (
        'No unmasked completion tokens after prompt boundary.'
    )
    full_inputs['labels'] = labels
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in full_inputs.items()}


def score_predictions_verifiable(
    refs: list[str],
    preds: list[str],
    finish_reasons: list[str],
    raw_refs: list[str] | None = None,
) -> dict:
    sw = sW = sc = sC = lw = lW = lc = lC = 0
    nl_edits = nl_chars = lb_tp = lb_p = lb_r = ot_edits = ot_chars = 0
    rewards, loops, budget_capped, non_terms, leaks, mods = [], 0, 0, 0, 0, 0
    for ref, pred, fr in zip(refs, preds, finish_reasons):
        pair = score_pair(pred, ref)
        st, ln = pair['strict'], pair['lenient']
        sw += st.word_edits; sW += st.ref_words
        sc += st.char_edits; sC += st.ref_chars
        lw += ln.word_edits; lW += ln.ref_words
        lc += ln.char_edits; lC += ln.ref_chars
        lb = score_line_breaks(pred, ref)
        nl_edits += int(lb['newline_char_edits'])
        nl_chars += int(lb['newline_ref_chars'])
        lb_tp += int(lb['matched_boundaries'])
        lb_p += int(lb['pred_lines'])
        lb_r += int(lb['ref_lines'])
        ot_edits += int(lb['order_tolerant_char_edits'])
        ot_chars += int(lb['order_tolerant_ref_chars'])
        ver = score_transcription(pred, ref, finish_reason=fr)
        rewards.append(ver.reward)
        loops += int(ver.loop_detected)
        if fr == 'length':
            budget_capped += 1
        non_terms += int(ver.non_terminated)
        leaks += ver.editorial_leak_count
        mods += ver.modernizations

    assert sw == sum(Levenshtein.distance(normalize_strict(r).split(), normalize_strict(p).split()) for r, p in zip(refs, preds))
    assert sc == sum(Levenshtein.distance(normalize_strict(r), normalize_strict(p)) for r, p in zip(refs, preds))
    if lb_p + lb_r:
        lb_f1 = (2.0 * lb_tp) / (lb_p + lb_r)
    else:
        lb_f1 = 1.0 if all(bool(normalize_preserve_newlines(p)) == bool(normalize_preserve_newlines(r)) for r, p in zip(refs, preds)) else 0.0
    out = {
        'strict_word_edits': sw,
        'strict_words': sW,
        'strict_wer_pct': round(100.0 * sw / sW, 2) if sW else 0.0,
        'strict_char_edits': sc,
        'strict_chars': sC,
        'strict_cer_pct': round(100.0 * sc / sC, 2) if sC else 0.0,
        'lenient_wer_pct': round(100.0 * lw / lW, 2) if lW else 0.0,
        'lenient_cer_pct': round(100.0 * lc / lC, 2) if lC else 0.0,
        'newline_cer_pct': round(100.0 * nl_edits / nl_chars, 2) if nl_chars else 0.0,
        'line_boundary_f1_pct': round(100.0 * lb_f1, 2),
        'order_tolerant_cer_pct': round(100.0 * ot_edits / ot_chars, 2) if ot_chars else 0.0,
        'mean_verifiable_reward': round(sum(rewards) / max(1, len(rewards)), 4),
        'loop_pages': loops,
        'budget_capped_pages': budget_capped,
        'non_terminated_pages': non_terms,
        'editorial_leaks': leaks,
        'modernizations': mods,
        'independent_recount_verified': True,
    }
    if raw_refs:
        rc = sum(Levenshtein.distance(normalize_strict(rr), normalize_strict(p)) for rr, p in zip(raw_refs, preds))
        rC = sum(len(normalize_strict(rr)) for rr in raw_refs)
        out['raw_strict_cer_pct'] = round(100.0 * rc / rC, 2) if rC else 0.0
    return out


def _detect_thinking_leak(raw_generated: str) -> bool:
    return any(tag in raw_generated for tag in ('<|channel>thought', '<|content_thinking|>', '<think>'))


@torch.no_grad()
def run_eval_panel(
    model,
    processor,
    rows: list[dict],
    panel_name: str,
    max_new_tokens: int | None = None,
    *,
    swap_images: bool = False,
) -> dict:
    model.eval()
    preds, refs, raw_refs, finish_reasons, per_side = [], [], [], [], []
    thinking_leak_pages = 0
    n_rows = len(rows)
    for idx, row in enumerate(tqdm(rows, desc=f'Eval {panel_name}', leave=False)):
        img_path = rows[(idx + 1) % n_rows]['image_path'] if (swap_images and n_rows > 1) else row['image_path']
        image = Image.open(img_path).convert('RGB')
        messages = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': DIPLOMATIC_PROMPT}]}]
        prompt = _apply_chat_nothink(processor, messages, add_generation_prompt=True)
        inputs = processor(text=[prompt], images=[image], return_tensors='pt').to(model.device)
        cap = int(max_new_tokens or MAX_EVAL_NEW_TOKENS)
        t0 = time.time()
        with (torch.amp.autocast('cuda', dtype=torch.bfloat16) if torch.cuda.is_available() else torch.no_grad()):
            out_ids = model.generate(**inputs, max_new_tokens=cap, do_sample=False)
        elapsed = round(time.time() - t0, 2)
        gen_ids = out_ids[:, inputs['input_ids'].shape[1]:]
        fr = 'length' if int(gen_ids.shape[1]) >= cap else 'stop'
        raw_gen = processor.batch_decode(gen_ids, skip_special_tokens=False)[0]
        if _detect_thinking_leak(raw_gen):
            thinking_leak_pages += 1
        gen = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
        ver = score_transcription(gen, row['text'], finish_reason=fr)
        lb = score_line_breaks(gen, row['text'])
        preds.append(gen)
        refs.append(row['text'])
        raw_refs.append(row.get('raw_text', row['text']))
        finish_reasons.append(fr)
        per_side.append({
            'record_id': row['record_id'],
            'document_id': row['document_id'],
            'elapsed_s': elapsed,
            'max_new_tokens': cap,
            'finish_reason': fr,
            'reward': ver.reward,
            'strict_cer': ver.cer_strict,
            'newline_cer': lb['newline_cer'],
            'order_tolerant_cer': lb['order_tolerant_cer'],
            'prediction': gen,
            'reference': row['text'],
        })
    scores = score_predictions_verifiable(refs, preds, finish_reasons, raw_refs=raw_refs)
    scores['thinking_leak_pages'] = thinking_leak_pages
    return {'panel': panel_name, 'sides': len(rows), 'scores': scores, 'predictions': per_side}


def _assert_highres_processor_config(model_dir: Path, hf_repo: str, reference_dir: Path | None = None) -> None:
    family = 'gemma4' if 'gemma-4' in hf_repo.lower() else 'inkling2'
    verify_exported_vlm_dir(
        model_dir,
        expected_soft_tokens=1120,
        check_gemma4_resolution=(family == 'gemma4'),
        check_nothink_template=True,
        model_family=family,
        reference_dir=reference_dir,
    )


def _build_arm_cache_key(cand_key: str, arm: str, spec, train_rows: list[dict], eval_rows: list[dict], use_4bit: bool) -> str:
    tpl_file = REPO_DIR / 'configs' / 'halo' / 'templates' / 'gemma4_htr_nothink.jinja'
    tpl_sha = hashlib.sha256(tpl_file.read_bytes()).hexdigest()[:12] if tpl_file.is_file() else 'hub'
    prompt_sha = hashlib.sha256(DIPLOMATIC_PROMPT.encode('utf-8')).hexdigest()[:12]
    payload = {
        'cand': cand_key, 'hf_repo': spec.hf_repo, 'hf_rev': spec.hf_revision, 'arm': arm, 'smoke': SMOKE_TEST_MODE,
        'ep': EPOCHS, 'r': LORA_RANK, 'alpha': LORA_ALPHA, 'drop': LORA_DROPOUT, 'wd': WEIGHT_DECAY,
        'warmup': WARMUP_RATIO, 'lr': LEARNING_RATE, 'vlr': VISION_LR, 'ga': GRAD_ACCUM_STEPS, 'seq': MAX_SEQ_LEN,
        'max_new': MAX_EVAL_NEW_TOKENS, 'seed': SEED,
        'oof': [globals().get('OOF_NUM_FOLDS', 1), globals().get('OOF_FOLD_IDX', 0)],
        'use_4bit': use_4bit, 'git_sha': globals().get('REPO_GIT_SHA', 'unknown'), 'tpl': tpl_sha, 'prompt': prompt_sha,
        'max_px': spec.max_pixels, 'lora_targets': list(spec.lora_target_modules), 'vis_pats': list(spec.vision_trainable_patterns),
        'train_sig': [[r.get('record_id', ''), r.get('image_sha256', ''), hashlib.sha256(r.get('text', '').encode()).hexdigest()[:12], r.get('sham_donor_record_id', '')] for r in train_rows],
        'eval_sig': [[r.get('record_id', ''), r.get('image_sha256', ''), hashlib.sha256(r.get('text', '').encode()).hexdigest()[:12]] for r in eval_rows],
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode('utf-8')).hexdigest()[:16]


def _merge_saved_adapter(spec, processor, adapter_dir: Path, merged_dir: Path, use_4bit: bool, live_model=None) -> str | None:
    merged_dir.parent.mkdir(parents=True, exist_ok=True)
    if not use_4bit and live_model is not None:
        merged_model = live_model.merge_and_unload()
        merged_model.save_pretrained(merged_dir, safe_serialization=True)
        processor.save_pretrained(merged_dir)
        del merged_model
        _assert_highres_processor_config(merged_dir, spec.hf_repo, reference_dir=adapter_dir)
        return str(merged_dir)
    try:
        host_ram_gb = (os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES')) / (1024 ** 3)
    except Exception:
        host_ram_gb = None
    if host_ram_gb is not None and spec.params_b * 3.0 > host_ram_gb:
        print(f'[SKIP MERGE] Host RAM {host_ram_gb:.1f} GB < ~{spec.params_b * 3.0:.0f} GB needed; adapter saved at {adapter_dir}.')
        return None
    cpu_kw = {'device_map': 'cpu', 'torch_dtype': torch.bfloat16, 'trust_remote_code': spec.trust_remote_code}
    if spec.hf_revision:
        cpu_kw['revision'] = spec.hf_revision
    cpu_base = load_vlm_model(spec.hf_repo, cpu_kw)
    merged_model = PeftModel.from_pretrained(cpu_base, str(adapter_dir)).merge_and_unload()
    merged_model.save_pretrained(merged_dir, safe_serialization=True)
    processor.save_pretrained(merged_dir)
    del merged_model, cpu_base
    _assert_highres_processor_config(merged_dir, spec.hf_repo, reference_dir=adapter_dir)
    return str(merged_dir)


def run_candidate_arm(cand_key: str, arm: str, arm_train_rows: list[dict], val_slice: list[dict], test_slice: list[dict]) -> dict:
    spec = COMPETITION_CANDIDATES[cand_key]
    use_4bit = should_use_4bit(spec, VRAM_GB, arm)
    eval_target = test_slice if (FINAL_FREEZE_EVAL_TEST_PUBLIC and test_slice) else val_slice
    eval_split_name = 'public_test' if (FINAL_FREEZE_EVAL_TEST_PUBLIC and test_slice) else 'val'
    out_dir = OUTPUT_ROOT / f'{cand_key}-{arm}'
    cache_key = _build_arm_cache_key(cand_key, arm, spec, arm_train_rows, eval_target, use_4bit)
    run_dir = out_dir / cache_key
    merged_dir = WORK_ROOT / 'merged' / f'{cand_key}-{arm}-{cache_key}'
    out_dir.mkdir(parents=True, exist_ok=True); run_dir.mkdir(parents=True, exist_ok=True)
    score_file = out_dir / f'{eval_split_name}_eval_scores_{cache_key}.json'
    if score_file.is_file():
        cached = json.loads(score_file.read_text(encoding='utf-8'))
        if arm == 'baseline_zeroshot':
            print(f'[SKIP] Cache hit {cand_key}-{arm} ({cache_key}): {score_file}')
            return cached
        a_dir = cached.get('adapter_dir')
        if not a_dir or not Path(a_dir).is_dir():
            print(f'[WARN] Cached adapter_dir missing on disk; invalidating stale cache and re-running.')
        else:
            m_dir = cached.get('merged_hf_dir')
            if not m_dir or not Path(m_dir).is_dir():
                print(f'[RECOVER] Rebuilding missing merged_hf_dir from {a_dir} -> {merged_dir}')
                proc = load_processor_fullres({'hf_repo': spec.hf_repo, 'hf_revision': spec.hf_revision, 'max_pixels': spec.max_pixels, 'trust_remote_code': spec.trust_remote_code})
                try:
                    cached['merged_hf_dir'] = _merge_saved_adapter(spec, proc, Path(a_dir), merged_dir, use_4bit)
                    score_file.write_text(json.dumps(cached, indent=2), encoding='utf-8')
                except Exception as rec_exc:
                    cached['merged_hf_dir'] = None
                    print(f'[WARN] Cached merge recovery failed: {rec_exc}')
            print(f'[SKIP] Cache hit {cand_key}-{arm} ({cache_key}): {score_file}')
            return cached

    print(f'\n================ {cand_key} ({spec.hf_repo}@{spec.hf_revision}) | ARM={arm} | EVAL={eval_split_name} ================')
    processor = load_processor_fullres({'hf_repo': spec.hf_repo, 'hf_revision': spec.hf_revision, 'max_pixels': spec.max_pixels, 'trust_remote_code': spec.trust_remote_code})

    if use_4bit and spec.quant_method == 'modelopt':
        raise RuntimeError(f'Candidate {spec.hf_repo} uses {spec.quant_method} quantization; cannot use 4-bit NF4 QLoRA.')

    load_kwargs = {'device_map': {'': 0} if use_4bit else 'auto', 'torch_dtype': torch.bfloat16, 'trust_remote_code': spec.trust_remote_code}
    if spec.hf_revision:
        load_kwargs['revision'] = spec.hf_revision
    if 'gemma-4' in spec.hf_repo.lower():
        load_kwargs['attn_implementation'] = 'sdpa'
    if use_4bit:
        print(f'VRAM {VRAM_GB} GB: loading {cand_key} ({arm}) in 4-bit NF4 (BF16 compute/storage)')
        bnb_kw = dict(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_storage=torch.bfloat16)
        if 'gemma-4' not in spec.hf_repo.lower():
            bnb_kw['llm_int8_skip_modules'] = list(spec.vision_trainable_patterns)
        load_kwargs['quantization_config'] = BitsAndBytesConfig(**bnb_kw)

    model = load_vlm_model(spec.hf_repo, load_kwargs)

    if arm == 'baseline_zeroshot':
        eval_res = run_eval_panel(model, processor, eval_target, panel_name=f'{cand_key}-zeroshot-{eval_split_name}')
        payload = {
            'candidate': cand_key, 'hf_repo': spec.hf_repo, 'arm': arm, 'eval_split': eval_split_name,
            'cache_key': cache_key, 'used_4bit': use_4bit, 'per_side_cer': [p['strict_cer'] for p in eval_res['predictions']], 'scores': eval_res['scores'],
        }
        (out_dir / f'predictions_{eval_split_name}_{cache_key}.json').write_text(json.dumps(eval_res['predictions'], ensure_ascii=False, indent=2), encoding='utf-8')
        score_file.write_text(json.dumps(payload, indent=2), encoding='utf-8')
        print(f'[ZERO-SHOT {cand_key} | {eval_split_name}]', payload['scores'])
        del model; gc.collect(); torch.cuda.empty_cache()
        return payload

    gc_kwargs = {'use_reentrant': False}
    if use_4bit:
        model = prepare_model_for_kbit_training(
            model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs=gc_kwargs
        )
    else:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs=gc_kwargs)
        model.enable_input_require_grads()

    try:
        vision_modules = resolve_vision_modules_to_save(
            (n for n, _ in model.named_modules()),
            spec.vision_trainable_patterns,
        )
    except RuntimeError as exc:
        top_names = sorted({n.split('.')[0] for n, _ in model.named_modules() if n})[:20]
        print('Module inspection failed. Top-level modules:', top_names)
        raise exc
    print('Resolved trainable vision modules:', vision_modules)
    lora_targets = 'all-linear' if spec.lora_target_modules == ('all-linear',) else list(spec.lora_target_modules)
    lora_kw = {
        'r': LORA_RANK, 'lora_alpha': LORA_ALPHA, 'lora_dropout': LORA_DROPOUT,
        'target_modules': lora_targets, 'bias': 'none', 'task_type': 'CAUSAL_LM',
    }
    if not use_4bit:
        lora_kw.update(exclude_modules=vision_modules, modules_to_save=vision_modules)
    model = get_peft_model(model, LoraConfig(**lora_kw))

    vision_params, lora_params = [], []
    for pname, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if any(vp in pname for vp in spec.vision_trainable_patterns) or 'modules_to_save' in pname:
            vision_params.append(param)
        else:
            lora_params.append(param)
    n_vision_trainable = sum(p.numel() for p in vision_params)
    n_lora_trainable = sum(p.numel() for p in lora_params)
    print(f'Verified trainable params -> vision: {n_vision_trainable:,} | LoRA: {n_lora_trainable:,}')
    assert n_vision_trainable > 0, 'FATAL: 0 vision parameters are trainable!'

    optimizer = torch.optim.AdamW([
        {'params': lora_params, 'lr': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY},
        {'params': vision_params, 'lr': VISION_LR, 'weight_decay': WEIGHT_DECAY},
    ])
    total_opt_steps = max(1, math.ceil(len(arm_train_rows) * EPOCHS / GRAD_ACCUM_STEPS))
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, int(total_opt_steps * WARMUP_RATIO)),
        num_training_steps=total_opt_steps,
    )

    rng = random.Random(SEED)
    best_val_cer = float('inf')
    best_epoch = 0
    best_trainable_state = None
    train_log = []
    skipped_oom = 0
    discarded_oom_microbatches = 0
    skipped_overlength = 0
    verified_step1_grads = False
    realized_image_tokens = None
    opt_step = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        order = list(arm_train_rows)
        rng.shuffle(order)
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        n_batches = 0
        accum_count = 0

        for row in tqdm(order, desc=f'{cand_key}-{arm} ep{epoch}/{EPOCHS}'):
            try:
                image = Image.open(row['image_path']).convert('RGB')
                batch = build_training_batch(processor, image, row['text'], model.device)
                if batch is None:
                    skipped_overlength += 1
                    continue
                if realized_image_tokens is None:
                    tok = getattr(processor, 'tokenizer', processor)
                    img_tid = getattr(tok, 'image_token_id', None)
                    if img_tid is not None:
                        realized_image_tokens = int((batch['input_ids'] == img_tid).sum().item())
                        print(f'Batch 0 realized image_tokens_per_page={realized_image_tokens}')
                        if 'gemma-4' in spec.hf_repo.lower() and realized_image_tokens <= 280:
                            raise RuntimeError(f'Gemma 4 vision tokens collapsed to {realized_image_tokens} (<= 280).')
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    out = model(**batch)
                    loss = out.loss / GRAD_ACCUM_STEPS
                loss.backward()
                if not verified_step1_grads:
                    has_live_vision_grad = any(
                        p.grad is not None and p.grad.abs().sum().item() > 0 for p in vision_params
                    )
                    if not has_live_vision_grad:
                        raise RuntimeError('Live step-1 vision gradient check failed: vision grad norm is zero.')
                    verified_step1_grads = True
                running_loss += float(out.loss.detach().cpu())
                n_batches += 1
                accum_count += 1
            except torch.cuda.OutOfMemoryError:
                skipped_oom += 1
                discarded_oom_microbatches += accum_count
                optimizer.zero_grad(set_to_none=True)
                accum_count = 0
                torch.cuda.empty_cache()
                continue

            if accum_count >= GRAD_ACCUM_STEPS:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                accum_count = 0
                opt_step += 1
                if globals().get('CHECKPOINT_EVERY_OPT_STEPS', 0) > 0 and opt_step % int(CHECKPOINT_EVERY_OPT_STEPS) == 0:
                    # Manual salvage snapshot only (weights + processor; no optimizer/scheduler state or auto-resume)
                    model.save_pretrained(run_dir / 'adapter_last')
                    processor.save_pretrained(run_dir / 'adapter_last')

        if accum_count > 0:
            scale = GRAD_ACCUM_STEPS / accum_count
            for param in model.parameters():
                if param.grad is not None:
                    param.grad.mul_(scale)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        if n_batches == 0:
            raise RuntimeError(f'{cand_key}-{arm} epoch {epoch} completed 0 valid batches (oom={skipped_oom}, overlength={skipped_overlength}).')
        if skipped_oom > max(3, int(0.1 * len(order))):
            raise RuntimeError(f'{cand_key}-{arm} exceeded OOM threshold: skipped_oom={skipped_oom}/{len(order)}.')

        avg_loss = round(running_loss / max(1, n_batches), 4)
        val_eval = run_eval_panel(model, processor, val_slice, panel_name=f'{cand_key}-{arm}-val-ep{epoch}')
        val_cer = val_eval['scores']['strict_cer_pct']
        train_log.append({
            'epoch': epoch,
            'train_loss': avg_loss,
            'skipped_oom': skipped_oom,
            'skipped_overlength': skipped_overlength,
            'val_scores': val_eval['scores'],
        })
        print(f'Epoch {epoch} | loss={avg_loss} | val={val_eval["scores"]}')

        if val_cer <= best_val_cer:
            best_val_cer = val_cer
            best_epoch = epoch
            best_trainable_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
            adapter_dir = run_dir / 'adapter_best'
            model.save_pretrained(adapter_dir)
            processor.save_pretrained(adapter_dir)

    if best_trainable_state is not None:
        with torch.no_grad():
            for n, p in model.named_parameters():
                if n in best_trainable_state:
                    p.copy_(best_trainable_state[n].to(device=p.device, dtype=p.dtype))
        print(f'Restored best-val checkpoint from epoch {best_epoch} (val strict CER={best_val_cer}%)')

    final_eval = run_eval_panel(model, processor, eval_target, panel_name=f'{cand_key}-{arm}-{eval_split_name}')
    pred_bytes = json.dumps(final_eval['predictions'], ensure_ascii=False, indent=2).encode('utf-8')
    (out_dir / f'predictions_{eval_split_name}_{cache_key}.json').write_bytes(pred_bytes)
    swap_delta = None
    if arm == 'real' and len(eval_target) >= 4:
        swap_subset = eval_target[:min(16, len(eval_target))]
        swap_eval = run_eval_panel(model, processor, swap_subset, panel_name=f'{cand_key}-real-swap-check', swap_images=True)
        matched_preds = final_eval['predictions'][:len(swap_subset)]
        matched_sub = score_predictions_verifiable(
            [p['reference'] for p in matched_preds],
            [p['prediction'] for p in matched_preds],
            [p['finish_reason'] for p in matched_preds],
        )['strict_cer_pct']
        swap_delta = round(swap_eval['scores']['strict_cer_pct'] - matched_sub, 2)
        print(f'Image-swap check ({len(swap_subset)} pages): swap CER={swap_eval["scores"]["strict_cer_pct"]}% (delta={swap_delta:+.2f} pp)')

    payload = {
        'candidate': cand_key, 'hf_repo': spec.hf_repo, 'arm': arm, 'eval_split': eval_split_name, 'cache_key': cache_key,
        'train_sides': len(arm_train_rows), 'vision_trainable_params': n_vision_trainable, 'lora_trainable_params': n_lora_trainable,
        'image_tokens_per_page': realized_image_tokens, 'used_4bit_qlora': use_4bit, 'skipped_overlength': skipped_overlength,
        'discarded_oom_microbatches': discarded_oom_microbatches, 'best_epoch': best_epoch, 'train_log': train_log,
        'predictions_sha256': hashlib.sha256(pred_bytes).hexdigest(), 'per_side_cer': [p['strict_cer'] for p in final_eval['predictions']],
        'swap_cer_degradation': swap_delta, 'scores': final_eval['scores'], 'adapter_dir': str(run_dir / 'adapter_best'), 'merged_hf_dir': None,
    }
    score_file.write_text(json.dumps(payload, indent=2), encoding='utf-8')

    try:
        if use_4bit:
            del model, optimizer, scheduler; gc.collect(); torch.cuda.empty_cache()
            model = optimizer = scheduler = None
        payload['merged_hf_dir'] = _merge_saved_adapter(
            spec, processor, run_dir / 'adapter_best', merged_dir, use_4bit, live_model=(None if use_4bit else model)
        )
        score_file.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    except Exception as merge_exc:
        print(f'[WARN] Post-eval merge failed (scores & adapter already saved): {merge_exc}')
    print(f'[DONE {cand_key}-{arm} | {eval_split_name}]', payload['scores'])

    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()
    return payload

In [ ]:
# === Step 3: Dev Validation Only (test_public stays frozen until final eval) ===
def _stratified_eval_slice(rows: list[dict], max_pages: int | None, seed: int) -> list[dict]:
    if not max_pages or len(rows) <= max_pages:
        return list(rows)
    rng = random.Random(seed)
    by_group = {}
    for row in rows:
        by_group.setdefault(row.get('document_id', ''), []).append(row)
    gkeys = sorted(by_group, key=lambda gid: (gid.split(':', 1)[0], hashlib.sha256(f'{seed}:{gid}'.encode()).hexdigest()))
    by_inst = {}
    for gid in gkeys:
        rng.shuffle(by_group[gid])
        by_inst.setdefault(gid.split(':', 1)[0], []).append(gid)
    picked = []
    while len(picked) < max_pages and any(by_inst.values()):
        for inst in sorted(by_inst):
            if not by_inst[inst]:
                continue
            gid = by_inst[inst][0]
            picked.append(by_group[gid].pop(0))
            if not by_group[gid]:
                by_inst[inst].pop(0)
            else:
                by_inst[inst].append(by_inst[inst].pop(0))
            if len(picked) >= max_pages:
                break
    return picked


def _fold_key(row: dict) -> str:
    return str(row['writer_key']) if row.get('writer_attributed') else str(row.get('document_id', ''))


def _select_doc_disjoint_pilot(rows: list[dict], max_pages: int | None, seed: int) -> list[dict]:
    if max_pages is None or len(rows) <= max_pages:
        return list(rows)
    by_group = {}
    for r in rows:
        by_group.setdefault(_fold_key(r), []).append(r)
    gkeys = sorted(by_group, key=lambda gid: hashlib.sha256(f'{seed}:{gid}'.encode()).hexdigest())
    picked = []
    for gid in gkeys:
        if picked and len(picked) + len(by_group[gid]) > max_pages:
            continue
        picked.extend(by_group[gid])
        if len(picked) >= max_pages:
            break
    return picked or list(rows[:max_pages])


def _assign_balanced_folds(rows: list[dict], num_folds: int, seed: int) -> dict[str, int]:
    by_key = {}
    for r in rows:
        by_key.setdefault(_fold_key(r), []).append(r)
    loads = [0] * num_folds
    fold_map = {}
    for key in sorted(by_key, key=lambda k: (-len(by_key[k]), hashlib.sha256(f'{seed}:{k}'.encode()).hexdigest())):
        best_fold = min(range(num_folds), key=lambda f: loads[f])
        fold_map[key] = best_fold
        loads[best_fold] += len(by_key[key])
    return fold_map


val_slice = _stratified_eval_slice(val_rows, VAL_EVAL_MAX_PAGES, SEED)
if OOF_NUM_FOLDS > 1:
    fold_map = _assign_balanced_folds(train_rows, OOF_NUM_FOLDS, SEED)
    active_train_rows = [r for r in train_rows if fold_map[_fold_key(r)] != OOF_FOLD_IDX]
else:
    active_train_rows = train_rows
pilot_train_rows = _select_doc_disjoint_pilot(active_train_rows, SHAM_MAX_PRESENTATIONS, SEED)
sham_rows = make_sham_derangement(pilot_train_rows, seed=SEED)
dev_test_slice = test_public_rows if FINAL_FREEZE_EVAL_TEST_PUBLIC else []
eval_slice = dev_test_slice if (FINAL_FREEZE_EVAL_TEST_PUBLIC and dev_test_slice) else val_slice

ALL_RESULTS = {}
for cand_key in CANDIDATE_KEYS:
    ALL_RESULTS[cand_key] = {}
    for arm in ARMS:
        arm_rows = sham_rows if arm == 'sham_short' else (
            pilot_train_rows if (arm == 'real' and SMOKE_TEST_MODE) else active_train_rows
        )
        ALL_RESULTS[cand_key][arm] = run_candidate_arm(cand_key, arm, arm_rows, val_slice, dev_test_slice)

summary = []
for cand_key, arms in ALL_RESULTS.items():
    z_res = arms.get('baseline_zeroshot', {})
    r_res = arms.get('real', {})
    s_res = arms.get('sham_short', {})
    z, r, s = z_res.get('scores', {}), r_res.get('scores', {}), s_res.get('scores', {})
    ci_vs_base = None
    if z_res.get('per_side_cer') and r_res.get('per_side_cer'):
        diffs = [rv - zv for rv, zv in zip(r_res['per_side_cer'], z_res['per_side_cer'])]
        eval_sub = eval_slice[:len(diffs)]
        cids = [row['document_id'] for row in eval_sub]
        wts = [max(1, len(normalize_strict(row['text']))) for row in eval_sub]
        ci_vs_base = paired_bootstrap_ci(diffs, cids, weights=wts, seed=SEED)
    summary.append({
        'candidate': cand_key,
        'eval_split': 'test_public' if FINAL_FREEZE_EVAL_TEST_PUBLIC else 'val',
        'zeroshot_wer_cer': f"{z.get('strict_wer_pct')}% / {z.get('strict_cer_pct')}%" if z else 'n/a',
        'zeroshot_raw_cer': z.get('raw_strict_cer_pct'),
        'sham_wer_cer': f"{s.get('strict_wer_pct')}% / {s.get('strict_cer_pct')}%" if s else 'n/a',
        'real_wer_cer': f"{r.get('strict_wer_pct')}% / {r.get('strict_cer_pct')}%" if r else 'n/a',
        'real_newline_f1': r.get('line_boundary_f1_pct'),
        'real_order_tol_cer': r.get('order_tolerant_cer_pct'),
        'real_swap_cer_delta': r_res.get('swap_cer_degradation'),
        'ci_real_minus_zeroshot_cer': ci_vs_base,
        'real_reward': r.get('mean_verifiable_reward'),
        'real_loops': r.get('loop_pages'),
    })
(OUTPUT_ROOT / 'colab_val_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))

In [ ]:
# === Optional: Generate & Score Stage 2 Rollouts for Halo VLM SMPO (chosen_sft_ratio: 1.0) ===
if GENERATE_OFFLINE_GRPO_ROLLOUTS:
    cand_key = CANDIDATE_KEYS[0]
    spec = COMPETITION_CANDIDATES[cand_key]
    real_res = ALL_RESULTS.get(cand_key, {}).get('real', {}) if 'ALL_RESULTS' in globals() else {}
    merged_dir = Path(real_res['merged_hf_dir']) if real_res.get('merged_hf_dir') else (OUTPUT_ROOT / f'{cand_key}-real' / 'merged_hf')
    adapter_dir = Path(real_res['adapter_dir']) if real_res.get('adapter_dir') else (OUTPUT_ROOT / f'{cand_key}-real' / 'adapter_best')
    assert merged_dir.is_dir() or adapter_dir.is_dir(), f'Train Stage 1 real checkpoint first under {OUTPUT_ROOT}'

    processor = load_processor_fullres({'hf_repo': spec.hf_repo, 'hf_revision': spec.hf_revision, 'max_pixels': spec.max_pixels, 'trust_remote_code': spec.trust_remote_code})
    rollout_4bit = should_use_4bit(spec, VRAM_GB, 'baseline_zeroshot')
    if rollout_4bit and spec.quant_method == 'modelopt':
        raise RuntimeError(f'{spec.hf_repo} uses {spec.quant_method} and cannot use 4-bit bitsandbytes rollouts.')
    load_target = str(merged_dir) if merged_dir.is_dir() else spec.hf_repo
    rollout_load_kwargs = {'device_map': 'auto', 'torch_dtype': torch.bfloat16, 'trust_remote_code': spec.trust_remote_code}
    if not merged_dir.is_dir() and spec.hf_revision:
        rollout_load_kwargs['revision'] = spec.hf_revision
    if 'gemma-4' in spec.hf_repo.lower():
        rollout_load_kwargs['attn_implementation'] = 'sdpa'
    if rollout_4bit:
        rollout_load_kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            llm_int8_skip_modules=list(spec.vision_trainable_patterns),
        )
    base_model = load_vlm_model(load_target, rollout_load_kwargs)
    model = base_model if merged_dir.is_dir() else PeftModel.from_pretrained(base_model, str(adapter_dir))
    model.eval()

    ROLLOUT_TEMPS = (0.3, 0.7, 0.7, 1.0)
    trained_rows = pilot_train_rows if ('pilot_train_rows' in globals() and SMOKE_TEST_MODE) else active_train_rows
    trained_docs = {r['document_id'] for r in trained_rows}
    trained_keys = {_fold_key(r) for r in trained_rows}
    oof_rows = [r for r in train_rows if r['document_id'] not in trained_docs and _fold_key(r) not in trained_keys]
    fold_tag = f'fold_{OOF_FOLD_IDX}_of_{OOF_NUM_FOLDS}' if OOF_NUM_FOLDS > 1 else ('smoke_doc_oof' if oof_rows else 'in_sample_train')
    if oof_rows and ROLLOUT_SOURCE == 'oof_train':
        rollout_rows = _stratified_eval_slice(oof_rows, GRPO_ROLLOUT_PROMPTS, SEED)
        is_oof = True
    elif ALLOW_IN_SAMPLE_TRAIN_ROLLOUTS:
        print('[WARN] No document-disjoint held-out train fold rows remain; using in-sample train rows because ALLOW_IN_SAMPLE_TRAIN_ROLLOUTS=True.')
        rollout_rows = _stratified_eval_slice(train_rows, GRPO_ROLLOUT_PROMPTS, SEED)
        is_oof = False
    else:
        raise RuntimeError(
            'Refusing in-sample or same-document Stage-2 rollouts without an out-of-fold train split (val/test_public stay excluded); '
            'set OOF_NUM_FOLDS >= 2 (or ALLOW_IN_SAMPLE_TRAIN_ROLLOUTS=True for debug) or pass held-out K-fold train shards to generate_halo_rollouts.py.'
        )
    out_jsonl = OUTPUT_ROOT / f'{cand_key}_scored_offline_grpo_rollouts.jsonl'
    pref_jsonl = OUTPUT_ROOT / f'{cand_key}_vlm_preferences.jsonl'
    pref_records = []
    rollout_thinking_leaks = 0
    with out_jsonl.open('w', encoding='utf-8') as fout:
        for row in tqdm(rollout_rows, desc='VLM Stage-2 Rollouts'):
            image = Image.open(row['image_path']).convert('RGB')
            messages = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': DIPLOMATIC_PROMPT}]}]
            prompt = _apply_chat_nothink(processor, messages, add_generation_prompt=True)
            inputs = processor(text=[prompt], images=[image], return_tensors='pt').to(model.device)
            tok = getattr(processor, 'tokenizer', processor)
            cap = recommended_max_new_tokens(len(tok(row['text'], add_special_tokens=False)['input_ids']), min_tokens=MAX_EVAL_NEW_TOKENS)
            completions, finish_reasons = [], []
            for gen_idx in range(GRPO_NUM_GENERATIONS):
                temp = ROLLOUT_TEMPS[gen_idx % len(ROLLOUT_TEMPS)]
                with torch.no_grad():
                    out_ids = model.generate(**inputs, max_new_tokens=cap, do_sample=True, temperature=temp, top_p=0.9)
                gen_ids = out_ids[:, inputs['input_ids'].shape[1]:]
                fr = 'length' if int(gen_ids.shape[1]) >= cap else 'stop'
                raw_gen = processor.batch_decode(gen_ids, skip_special_tokens=False)[0]
                if _detect_thinking_leak(raw_gen):
                    rollout_thinking_leaks += 1
                completions.append(processor.batch_decode(gen_ids, skip_special_tokens=True)[0])
                finish_reasons.append(fr)
            scored = score_group_completions(completions, row['text'], finish_reasons)
            fout.write(json.dumps({
                'record_id': row['record_id'],
                'image_path': row['image_path'],
                'prompt': DIPLOMATIC_PROMPT,
                'completions': completions,
                'rewards': [float(item['reward']) for item in scored],
                'scored_completions': scored,
            }, ensure_ascii=False) + '\n')
            pref_records.append({
                'record_id': row['record_id'],
                'document_id': row['document_id'],
                'writer_key': row['writer_key'],
                'image_sha256': row['image_sha256'],
                'image_dhash': row.get('image_dhash', ''),
                'image_path': row['image_path'],
                'prompt': DIPLOMATIC_PROMPT,
                'text': row['text'],
                'completions': completions,
                'finish_reasons': finish_reasons,
                'max_new_tokens_cap': cap,
                'split': row.get('split', 'train'),
                'source_fold': fold_tag,
                'is_out_of_fold': is_oof,
                'source_checkpoint': str(merged_dir if merged_dir.is_dir() else adapter_dir),
            })

    pref_pairs = build_preference_pairs(pref_records, min_reward_margin=0.05, prefer_gold_chosen=True)
    pref_jsonl.write_text(
        '\n'.join(json.dumps(pair, ensure_ascii=False) for pair in pref_pairs) + ('\n' if pref_pairs else ''),
        encoding='utf-8',
    )
    print(f'Wrote {len(pref_pairs)} gold-anchored Halo VLM preference pairs -> {pref_jsonl} (leaks={rollout_thinking_leaks})')

In [ ]:
# === Optional Local Quantized MLX Export (<= 24 GB Workstation Budget — 12B Only) ===
for cand_key in CANDIDATE_KEYS:
    spec = COMPETITION_CANDIDATES[cand_key]
    merged = ALL_RESULTS.get(cand_key, {}).get('real', {}).get('merged_hf_dir') or f'<merged_{cand_key}_hf_dir>'
    if spec.params_b > 15.0:
        print(f'# SKIP MLX 24 GB gate for {cand_key} ({spec.params_b}B > single-host 24 GB unified memory budget).')
        continue
    print(f'# {cand_key} ({spec.hf_repo}, est MLX peak ~{spec.est_mlx_peak_gb} GB):')
    print(f'python -m mlx_vlm convert --hf-path {merged} --mlx-path ./mlx/{cand_key}-q{spec.mlx_quant_bits} -q --q-bits {spec.mlx_quant_bits}')
    print(f'python scripts/verify_mlx_export.py ./mlx/{cand_key}-q{spec.mlx_quant_bits} --reference-dir {merged}')